# Bayesian Neural Networks Classifier

In [ ]:
""" Base code created on February 16, 2023 // @author: Sarah Shi """
""" Edited for alkanes work starting October 25th, 2023 // @editor: Ruth Tweedy """

In [ ]:
import numpy as np
import pandas as pd
import math 

import os
import copy
import time
import random

from sklearn.metrics import classification_report 
from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import matplotlib.colors as colors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['pdf.fonttype'] = 42


In [ ]:
# Parameters
input_file = "test_train_df.csv"
output_file = "test 2.csv"
alkanes = ['C25', 'C27', 'C29', 'C31', 'C33', 'C35']
run_description = "Full dataset, 180 validation"
is_run_interactive = True

## Defining Functions

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)

In [ ]:
def save_model_nn(model, optimizer, path, best_model_state):
    check_point = {'params': best_model_state,                            
                   'optimizer': optimizer.state_dict()}
    torch.save(check_point, path)

In [ ]:
def same_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python random module.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
SEED = 42
same_seeds(SEED)

In [ ]:
def predict_with_uncertainty(model, input_data, n_iterations=100):
    model.eval()
    output_list = []
    for i in range(n_iterations):
        with torch.no_grad():
            output = model(input_data)
            output_list.append(torch.nn.functional.softmax(output, dim=1).detach().cpu().numpy())

    output_list = np.array(output_list)
    
    # Calculate mean and standard deviation
    prediction_mean = output_list.mean(axis=0)
    prediction_stddev = output_list.std(axis=0)
    return prediction_mean, prediction_stddev

In [ ]:
def balance(train_data_x, train_data_y):

    oversample = RandomOverSampler(sampling_strategy='minority', random_state=42)

    # Resample the dataset
    x_balanced, y_balanced = oversample.fit_resample(train_data_x, train_data_y)

    df_resampled = pd.DataFrame(x_balanced)
    df_resampled['Mineral'] = y_balanced

    df_balanced = pd.DataFrame()
    for class_label in df_resampled['Mineral'].unique():
        df_class = df_resampled[df_resampled['Mineral'] == class_label]
        df_balanced = pd.concat([df_balanced, df_class.sample(n=1000, replace = True, random_state=42)])

    # Reset the index of the balanced dataframe
    df_balanced = df_balanced.reset_index(drop=True)
    train_data_x = df_balanced.iloc[:, :-1].to_numpy()
    train_data_y = df_balanced.iloc[:, -1].to_numpy()

    return train_data_x, train_data_y

In [ ]:
class LabelDataset(Dataset):
    def __init__(self, x, labels):
        if len(x.shape)==2:
            self.x = torch.from_numpy(x).type(torch.FloatTensor)
            self.labels = torch.from_numpy(labels.copy()).type(torch.LongTensor)
            #self.labels = torch.from_numpy(labels).type(torch.LongTensor)
        else:
            self.x = x.reshape(-1, x.shape[-1]) #dataset keeps the right shape for training
            self.labels = labels

    def __len__(self):
        return len(self.x) 
    
    def __getitem__(self, n): 
        return self.x[n], self.labels[n]

In [ ]:
class VariationalLayer(nn.Module):

    def __init__(self, in_features, out_features):

        super(VariationalLayer, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features))
        self.weight_rho = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.Tensor(out_features))
        self.bias_rho = nn.Parameter(torch.Tensor(out_features))
        
        self.softplus = nn.Softplus()
        self.reset_parameters()
        
    def reset_parameters(self):

        stdv = 1. / math.sqrt(self.weight_mu.size(1))
        self.weight_mu.data.uniform_(-stdv, stdv)
        self.weight_rho.data.uniform_(-stdv, stdv)
        self.bias_mu.data.uniform_(-stdv, stdv)
        self.bias_rho.data.uniform_(-stdv, stdv)
        
    def forward(self, input):

        weight_sigma = torch.log1p(torch.exp(self.weight_rho))
        bias_sigma = torch.log1p(torch.exp(self.bias_rho))
        
        weight_epsilon = torch.normal(mean=0., std=1., size=weight_sigma.size(), device=input.device)
        bias_epsilon = torch.normal(mean=0., std=1., size=bias_sigma.size(), device=input.device)
        
        weight_sample = self.weight_mu + weight_epsilon * weight_sigma
        bias_sample = self.bias_mu + bias_epsilon * bias_sigma
        
        output = F.linear(input, weight_sample, bias_sample)
        return output

    def kl_divergence(self):

        weight_sigma = torch.log1p(torch.exp(self.weight_rho))
        bias_sigma = torch.log1p(torch.exp(self.bias_rho))
        
        kl_div = -0.5 * torch.sum(1 + torch.log(weight_sigma.pow(2)) - self.weight_mu.pow(2) - weight_sigma.pow(2))
        kl_div += -0.5 * torch.sum(1 + torch.log(bias_sigma.pow(2)) - self.bias_mu.pow(2) - bias_sigma.pow(2))

        return kl_div

In [ ]:
class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim=6, classes=2, dropout_rate=0.1, hidden_layer_sizes=None):
        super(MultiClassClassifier, self).__init__()
        self.input_dim = input_dim
        self.classes = classes
        self.dropout_rate = dropout_rate

        # If hidden_layer_sizes is None or empty, initialize it as an empty list
        self.hls = hidden_layer_sizes or []

        def element(in_channel, out_channel, is_last=False):
            if not is_last:
                layers = [
                    nn.Linear(in_channel, out_channel),
                    nn.BatchNorm1d(out_channel),  # Add batch normalization
                    nn.LeakyReLU(0.02),
                    nn.Dropout(self.dropout_rate),  # Add dropout
                ]
            else:
                layers = [VariationalLayer(in_channel, out_channel)]
            return layers

        encoder = []
        if self.hls:
            # Construct hidden layers if hidden_layer_sizes is provided
            for i, size in enumerate(self.hls):
                if i == 0:
                    encoder += element(self.input_dim, size, is_last=(i==len(self.hls)-1))
                else:
                    encoder += element(self.hls[i-1], size, is_last=(i==len(self.hls)-1))
            last_hidden_size = self.hls[-1]
        else:
            # Skip hidden layers, use input_dim as last hidden layer size
            last_hidden_size = self.input_dim

        # Final layer from the last hidden layer (or input layer if no hidden layers) to the output
        encoder += [nn.Linear(last_hidden_size, self.classes)]

        self.encode = nn.Sequential(*encoder)
        self.apply(weights_init)

    def encoded(self, x):
        return self.encode(x)

    def forward(self, x):
        en = self.encoded(x)
        return en

    def predict(self, x):
        scores = self.forward(x)
        class_indices = scores.argmax(dim=1)
        return class_indices



class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_layer_sizes=None, dropout_rate=0.1):
        super(BinaryClassifier, self).__init__()

        self.input_dim = input_dim
        self.dropout_rate = dropout_rate
        self.hls = hidden_layer_sizes or []

        layers = []

        if self.hls:
            # Construct hidden layers if hidden_layer_sizes is provided
            for i, size in enumerate(self.hls):
                if i == 0:
                    layers.append(nn.Linear(input_dim, size))
                else:
                    layers.append(nn.Linear(self.hls[i-1], size))
                layers.append(nn.BatchNorm1d(size))
                layers.append(nn.LeakyReLU())
                layers.append(nn.Dropout(dropout_rate))
            last_hidden_size = self.hls[-1]
        else:
            # Skip hidden layers, use input_dim as the last hidden layer size
            last_hidden_size = self.input_dim

        # Final layer
        layers.append(nn.Linear(last_hidden_size, 2))  # 2 output neurons for binary classification

        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        x = self.layers(x)
        return x  # nn.CEL does not expect sigmoid activation

    def predict(self, x):
        scores = self.forward(x)
        class_indices = scores.argmax(dim=1)  # Choose the class with the higher score
        return class_indices


In [ ]:
def train_nn_binary(model, optimizer, label, train_loader, test_loader, n_epoch, criterion, patience=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')    
    model = model.to(device)
    
    avg_train_loss = []
    avg_test_loss = []
    best_test_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(n_epoch):
        model.train()
        t = time.time()
        train_loss = []
        for i, (data, labels) in enumerate(train_loader):
            x = data.to(device)
            y = labels.to(device)
            train_output = model(x)
            loss = criterion(train_output, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss.append(loss.detach().item())
        
        # Validation
        model.eval()
        test_loss = []
        with torch.no_grad():
            for i, (data, labels) in enumerate(test_loader):
                x = data.to(device)
                y = labels.to(device)
                test_output = model(x)
                loss = criterion(test_output, y)
                test_loss.append(loss.detach().item())

        # Logging
        avg_train = sum(train_loss) / len(train_loss)
        avg_test = sum(test_loss) / len(test_loss)
        avg_train_loss.append(avg_train)
        avg_test_loss.append(avg_test)
        
        training_time = time.time() - t
        print(f'[{epoch+1:03}/{n_epoch:03}] train_loss: {avg_train:.6f}, test_loss: {avg_test:.6f}, time: {training_time:.2f} s')

        # Early stopping
        if avg_test < best_test_loss:
            best_test_loss = avg_test
            best_epoch = epoch
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())  # Save the best model weights
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Validation loss hasn't improved for {patience} epochs. Stopping early.")
                break

    return train_output, test_output, avg_train_loss, avg_test_loss, best_test_loss, best_model_state


## Pre-Treated Dataframe Loading

In [ ]:
min_df = pd.read_csv(input_file).iloc[:, 1:]
code = pd.Categorical(min_df['wg']).codes
cat_lab = pd.Categorical(min_df['wg'])
min_df["sample_uid"] = np.arange(len(min_df))

targets = ['clr1', 'clr2', 'clr3', 'clr4', 'clr5', 'clr6']

train_data_x_df = min_df[targets][min_df['split'] == 'train']
train_data_y = code[train_data_x_df.index.to_numpy()] 
train_data_x = train_data_x_df.values

train_data_x_res_df = min_df[targets][(min_df['split'] == 'train') | (min_df['split'] == 'synthetic_train')]
train_data_y_res_code = code[train_data_x_res_df.index.to_numpy()]
train_data_x_res = train_data_x_res_df.values

test_data_x_df = min_df[targets][min_df['split'] == 'test']
test_data_y_code = code[test_data_x_df.index.to_numpy()] 
test_data_x = test_data_x_df

test_data_x_res_df = min_df[targets][(min_df['split'] == 'test') | (min_df['split'] == 'synthetic_test')]
test_data_y_res_code = code[test_data_x_res_df.index.to_numpy()] 
test_data_x_res = test_data_x_res_df

label = ['wg']

In [ ]:
def neuralnetwork_binary(test_data_x_res, train_data_x_res, train_data_y_res_code, test_data_y_res_code, name, hls_list, lr_list, wd, dr, batch_size):


    path_beg = os.getcwd() + '/'
    output_dir = ["nn_parametermatrix", "autoencoder_parametermatrix"] 
    for ii in range(len(output_dir)):
        if not os.path.exists(path_beg + output_dir[ii]):
            os.makedirs(path_beg + output_dir[ii], exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    feature_dataset = LabelDataset(train_data_x_res, train_data_y_res_code)
    test_dataset = LabelDataset(test_data_x_res.values, test_data_y_res_code)

    # Define data loaders
    feature_loader = DataLoader(feature_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
    np.savez('nn_parametermatrix/' + name + '_nn_features.npz', feature_loader=feature_loader, test_loader=test_loader)

    mapping = dict(zip(code, cat_lab))
    sort_mapping = dict(sorted(mapping.items(), key=lambda item: item[0]))

    train_losses_dict = {}
    test_losses_dict = {}
    input_size = len(feature_dataset.__getitem__(0)[0])
    best_test_loss = float('inf')
    best_model_state = None
    best_hidden_layer_size = None
    best_lr = None

    for hls in hls_list:
        for lr in lr_list: 
            
            same_seeds(SEED)
            print(f"Training with hidden layer sizes: {hls} and learning rate: {lr}")

            # Initialize model
            model = BinaryClassifier(input_dim=input_size, dropout_rate=dr, hidden_layer_sizes=hls).to(device)

            # Define loss function and optimizer
            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

            train_output, test_output, avg_train_loss, avg_test_loss, current_best_test_loss, current_best_model_state = train_nn_binary(model, optimizer, label, feature_loader, test_loader, epochs, criterion)
            
            if current_best_test_loss < best_test_loss:
                best_test_loss = current_best_test_loss
                best_hidden_layer_size = hls
                best_lr = lr
                best_model_state = current_best_model_state            

            if hls is None:
                hls_key = 'None'  # Use a string 'None' as a key if hls is None
            else:
                hls_key = tuple(hls)

            train_losses_dict[(hls_key)] = avg_train_loss
            test_losses_dict[(hls_key)] = avg_test_loss
            
            plt.figure(figsize = (8, 8))
            plt.plot(list(range(1, len(avg_train_loss)+1)), avg_train_loss, label = 'Train')
            plt.plot(list(range(1, len(avg_train_loss)+1)), avg_test_loss, label = 'Test')
            plt.xlabel('Epochs')
            plt.ylabel('Loss')
            plt.legend()
            plt.show()



            


    best_model = BinaryClassifier(input_dim=input_size, dropout_rate=dr, hidden_layer_sizes=best_hidden_layer_size) 
    best_model.load_state_dict(best_model_state)
    best_model.eval()

    with torch.no_grad():
        test_predictions = best_model(torch.Tensor(test_data_x_res.values)).cpu().numpy()
        train_predictions = best_model(torch.Tensor(train_data_x_res)).cpu().numpy()
        test_pred_classes = np.argmax(test_predictions, axis=1)
        train_pred_classes = np.argmax(train_predictions, axis=1)

    # Calculate classification metrics for the test dataset    
    test_report = classification_report(test_data_y_res_code, test_pred_classes, target_names=list(sort_mapping.values()), zero_division=0, output_dict=True)
    train_report = classification_report(train_data_y_res_code, train_pred_classes, target_names=list(sort_mapping.values()), zero_division=0, output_dict=True) # output_dict=True
    
    # Print the best kl_weight_decay value and test report
    print("Best hidden layer size:", best_hidden_layer_size)
    print("Best learning rate:", best_lr)
    print("Test report:", test_report)

    # Save the best model and other relevant information
    model_path = 'best_model.pt'
    save_model_nn(model, optimizer, model_path, best_model_state)
    np.savez('best_model_info.npz', test_report=test_report, train_report=train_report)

    return train_pred_classes, test_pred_classes, train_report, test_report, best_model_state, train_losses_dict, test_losses_dict, sort_mapping



## Neural Network Training!

In [ ]:
name = "test"

wd = 1e-3 
dr = 0.1 # 0.1 #0.25
n = 0.20

epochs = 2000
batch_size = 13 
hls_list = [None, [8], [16], [32], [64], [16, 4], [32, 8], [64, 16]]# [16, 8], [32, 16], [64, 32]]#, [64, 32, 16]] [8]
# hls_list=[None]
lr_list = [5e-5]#, 1e-5]

In [ ]:
mapping = dict(zip(code, cat_lab))
sort_mapping = dict(sorted(mapping.items(), key=lambda item: item[0]))

In [ ]:
train_pred_classes, test_pred_classes, train_report, test_report, best_model_state, train_losses_dict, test_losses_dict, sort_mapping \
              = neuralnetwork_binary(test_data_x_res, train_data_x_res, train_data_y_res_code, test_data_y_res_code, name, \
                   hls_list, lr_list, wd, dr, batch_size)


## Confusion Matrices

In [ ]:
train_data_res_df = train_data_x_res_df
train_data_res_df['code'] = train_data_y_res_code

In [ ]:
train_data_res_df

In [ ]:
train_data_res_df['pred'] = train_pred_classes
train_data_res_df
train_data_df_true = train_data_res_df.loc[train_data_res_df.index.isin(train_data_x_df.index)]

In [ ]:
train_data_df_true

In [ ]:
test_data_res_df = test_data_x_res_df
test_data_res_df['code'] = test_data_y_res_code
test_data_res_df['pred'] = test_pred_classes
test_data_res_df

In [ ]:
test_data_df_true = test_data_res_df.loc[test_data_res_df.index.isin(test_data_x_df.index)]
test_data_df_true

In [ ]:
cm_train = confusion_matrix(train_data_res_df['code'], train_data_res_df['pred'])

cm_train_true = confusion_matrix(train_data_df_true['code'], train_data_df_true['pred'])

cm_test = confusion_matrix(test_data_res_df['code'], test_data_res_df['pred'])

cm_test_true = confusion_matrix(test_data_df_true['code'], test_data_df_true['pred'])

class_labels = list(sort_mapping.values())

In [ ]:
cmap = plt.get_cmap('Blues')

In [ ]:
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

new_cmap = truncate_colormap(cmap, 0, 0.6)

In [ ]:
def plot_confusion_matrix(cm, class_labels, title):
    cm = cm.T  # Transpose the confusion matrix to flip the axes

    # Calculate the sum of rows and columns
    cm_with_sums = np.zeros((cm.shape[0] + 1, cm.shape[1] + 1), dtype=cm.dtype)
    cm_with_sums[:cm.shape[0], :cm.shape[1]] = cm
    cm_with_sums[-1, -1] = np.sum(cm)
    cm_with_sums[:-1, -1] = np.sum(cm, axis=1)
    cm_with_sums[-1, :-1] = np.sum(cm, axis=0)

    # Set parameters for font size and figure size
    plt.rcParams.update({'font.size': 14})  # Set the font size

    plt.figure(figsize=(5, 3))
    plt.imshow(cm_with_sums, cmap=new_cmap, interpolation='nearest')
    
 

    class_labels_with_sums = class_labels + ['Total']
    
    plt.xticks(ticks=range(len(class_labels_with_sums)), labels=class_labels_with_sums, rotation=45)
    plt.yticks(ticks=range(len(class_labels_with_sums)), labels=class_labels_with_sums)
    plt.xlabel('True')  # Flipped x and y axis labels
    plt.ylabel('Predicted')  # Flipped x and y axis labels
    plt.title(title)

    for i in range(len(class_labels_with_sums)):
        for j in range(len(class_labels_with_sums)):
            if i == len(class_labels_with_sums) - 1 or j == len(class_labels_with_sums) - 1:  # For sum cells
                cell_value = cm_with_sums[i, j]
                if i == len(class_labels_with_sums) - 1 and j == len(class_labels_with_sums) - 1:  # For the Total cell
                    total_diagonal_sum = np.sum(np.diag(cm))
                    if cell_value != 0:
                        percentage = total_diagonal_sum / cell_value
                        plt.text(j, i, f"{int(cell_value)}\n{percentage:.0%}", ha='center', va='center', color='black')
                    else:
                        plt.text(j, i, str(int(cell_value)), ha='center', va='center', color='black')
                elif i == len(class_labels_with_sums) - 1:  # For cells in the bottom row
                    pred_sum = cm_with_sums[-1, j]
                    diagonal_val = cm_with_sums[j, j]
                    if pred_sum != 0:
                        percentage = diagonal_val / pred_sum
                        plt.text(j, i, f"{int(cell_value)}\n{percentage:.0%}", ha='center', va='center', color='black')
                    else:
                        plt.text(j, i, str(int(cell_value)), ha='center', va='center', color='black')
                else:
                    true_sum = cm_with_sums[i, -1]
                    pred_sum = cm_with_sums[-1, j]
                    diagonal_val = cm_with_sums[i, i]
                    if true_sum != 0:
                        percentage = diagonal_val / true_sum
                        plt.text(j, i, f"{int(cell_value)}\n{percentage:.0%}", ha='center', va='center', color='black')
                    else:
                        plt.text(j, i, str(int(cell_value)), ha='center', va='center', color='black')
            else:
                plt.text(j, i, str(int(cm_with_sums[i, j])), ha='center', va='center', color='black')

    plt.colorbar()

plt.figure()
plot_confusion_matrix(cm_train, class_labels, 'NN Training')
plt.show()

plt.figure()
plot_confusion_matrix(cm_train_true, class_labels, 'NN Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(cm_test, class_labels, 'NN Validation')
plt.show()

plt.figure()
plot_confusion_matrix(cm_test_true, class_labels, 'NN Validation Excluding Synthetic')
plt.show()

## Analyzing Incorrect Data

In [ ]:
train_df = min_df[(min_df.split == 'train')]
test_df = min_df[min_df.split == 'test']

In [ ]:
train_df= pd.merge(train_df, train_data_res_df[['code', 'pred']], how='left', left_index=True, right_index=True)
train_df

test_df= pd.merge(test_df, test_data_res_df[['code', 'pred']], how='left', left_index=True, right_index=True)
test_df

In [ ]:
def separate_correct_incorrect(train_df, train_pred, train_actual, test_df, test_pred, test_actual):
    # Compare the predictions with the actual target variable
    incorrect_train_mask = train_pred != train_actual
    incorrect_test_mask = test_pred != test_actual
    
    incorrect_train = train_df[incorrect_train_mask]
    incorrect_test = test_df[incorrect_test_mask]
    
    incorrect = pd.concat([incorrect_train, incorrect_test])
    incorrect['Predicted'] = 'Incorrect'
    
    
    # Create a mask for correctly predicted rows
    correct_train_mask = train_pred == train_actual
    correct_test_mask = test_pred == test_actual
    
    correct_train = train_df[correct_train_mask]
    correct_test = test_df[correct_test_mask]
    
    correct = pd.concat([correct_train, correct_test])
    correct['Predicted'] = 'Correct'
    
    return incorrect, correct

In [ ]:
incorrect, correct = separate_correct_incorrect(train_df, train_data_df_true['pred'], train_data_df_true['code'], test_df, test_data_df_true['pred'], test_data_df_true['code'])#, min_df)

In [ ]:
def correct_incorrect_box_plots(model_type, correct, incorrect):

    # Variables to plot
    variables = alkanes

    # Create a figure with four subplots (Grassy Correct, Grassy Incorrect, Woody Correct, Woody Incorrect)
    fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharey=True, sharex=True)
    fig.suptitle(model_type, fontsize=30)

    # Plot box and whisker plots for Grassy Correct dataset
    grassy_correct_data = correct[correct['wg'] == 'Grassy'][variables]
    sns.boxplot(data=grassy_correct_data, ax=axes[0, 0], color='orange',)
    axes[0, 0].set_title(f'Grassy Correct, n={len(grassy_correct_data)}')
    axes[0, 0].set_ylabel('Values')
    #ax.set_title(f'{title} (n={n})')

    # Plot box and whisker plots for Grassy Incorrect dataset
    grassy_incorrect_data = incorrect[incorrect['wg'] == 'Grassy'][variables]
    sns.boxplot(data=grassy_incorrect_data, ax=axes[0, 1], color='red')
    axes[0, 1].set_title(f'Grassy Incorrect, n={len(grassy_incorrect_data)}')

    # Plot box and whisker plots for Woody Correct dataset
    woody_correct_data = correct[correct['wg'] == 'Woody'][variables]
    sns.boxplot(data=woody_correct_data, ax=axes[1, 0], color='lightgreen')
    axes[1, 0].set_title(f'Woody Correct, n={len(woody_correct_data)}')
    axes[1, 0].set_ylabel('Values')
    axes[1, 0].set_xlabel('Variables')

    # Plot box and whisker plots for Woody Incorrect dataset
    woody_incorrect_data = incorrect[incorrect['wg'] == 'Woody'][variables]
    sns.boxplot(data=woody_incorrect_data, ax=axes[1, 1], color='darkgreen')
    axes[1, 1].set_title(f'Woody Incorrect, n={len(woody_incorrect_data)}')
    axes[1, 1].set_xlabel('Variables')

    # Adjust spacing between the plots
    plt.tight_layout()

    # Show the plots
    plt.show()

In [ ]:
correct_incorrect_box_plots("NN Real Data", correct, incorrect)